# Voice Studio AI — Kaggle GPU

**Tạo giọng nói bằng AI trên GPU miễn phí của Kaggle.** Không chỉ "gõ chữ → ra tiếng":
đây là một **studio thu nhỏ** để *thiết kế* và *nhân bản* giọng nói, lưu thành kho giọng
tái sử dụng, dựng podcast/audiobook nhiều giọng — rồi xuất file.

| Tab | Làm được gì |
|---|---|
| **1 · Design** | Mô tả giọng bằng chữ → tạo một giọng **chưa từng tồn tại** (không cần clip) |
| **2 · Clone** | Tải clip 3–15 giây → nhân bản giọng + điều khiển style |
| **3 · Bank** | Kho giọng: xem / đổi / xóa, **export/import .zip để giữ giọng qua session** |
| **4 · Script** | Kịch bản nhiều giọng → podcast / audiobook / hội thoại |
| **5 · Export** | Tải WAV/MP3 + nén toàn bộ ra .zip |

**Cách dùng:** chọn *Accelerator = GPU (T4 x2)*, *Internet = ON* → **Run All** →
bấm vào link Gradio ở cell cuối.

**Engine:** VieNeu v3-Turbo (tiếng Việt) hoặc VoxCPM2 (Design + Clone + Style, 30 ngôn ngữ).
Đổi engine ở cell `CFG` bên dưới. Cả hai **Apache-2.0**.

> **Author: Elainabaka** © 2026 — thiết kế & dựng notebook Voice Studio AI cho cộng đồng.
> Code MIT · model giữ license Apache-2.0 của tác giả gốc. Xem mục Credit ở cuối notebook.
> **Chỉ clone giọng bạn có quyền sử dụng.**


## 🧭 Chỉ đường

1. **Cấu hình & GPU** — đổi `ENGINE` nếu muốn.
2. **Cài thư viện** — chỉ cài engine đang chọn (tránh xung đột dependency).
3. **Nạp Voice Studio** — giải nén gói `studio/` ra đĩa rồi import.
4. **Nạp model + test nhanh** — hot-swap, chỉ 1 model vào VRAM.
5. **Gradio Studio** — giao diện 5 tab để dùng ngay.


In [ ]:
# ===== 1 · CẤU HÌNH & KIỂM TRA GPU =====
ENGINE = "vieneu"   # "vieneu" (tiếng Việt) | "voxcpm" (Design+Clone+Style, 30 ngôn ngữ)

import torch
assert torch.cuda.is_available(), "Bật Accelerator = GPU (T4 x2) trong Settings rồi chạy lại!"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Engine đã chọn: {ENGINE}")


In [ ]:
# ===== 2 · CÀI THƯ VIỆN (chỉ engine đang chọn) =====
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("gradio>=4.44", "soundfile", "librosa")
if ENGINE == "vieneu":
    pip("vieneu")                       # VieNeu v3-Turbo (Apache-2.0)
elif ENGINE == "voxcpm":
    pip("voxcpm")                       # VoxCPM2 (Apache-2.0)
    try:                                # flash-attn = tăng tốc (tùy chọn), lỗi cũng không sao
        pip("flash-attn==2.7.4.post1", "--no-build-isolation")
    except Exception as e:
        print("flash-attn bỏ qua (không bắt buộc):", e)
print("Cài xong.")


In [ ]:
# ===== 3 · NẠP VOICE STUDIO (giải nén gói studio/ ra đĩa) =====
import base64, os, sys, json

PKG = "/kaggle/working/voice_studio_pkg"
os.makedirs(PKG, exist_ok=True)
_FILES = json.loads(r'''{
"studio/__init__.py": "IiIiVm9pY2UgU3R1ZGlvIEFJIC0gY29yZSBwYWNrYWdlIChLYWdnbGUgR1BVKS4KCk1vZHVsZXMKLS0tLS0tLQpjb25maWcgICAgQ0ZHICsgZW5zdXJlX2RpcnMKYXVkaW8gICAgIGxvYWQvc2F2ZS9yZXNhbXBsZS9tb25vL3RyaW0vbG91ZG5lc3MKdm9pY2ViYW5rIFZvaWNlUHJvZmlsZSArIGJhbmsgc2F2ZS9sb2FkL2V4cG9ydC9pbXBvcnQgIChUUlUgQ09UIC0gbHV1IGdpb25nIHZ1b3Qgc2Vzc2lvbikKcmVuZGVyICAgIG11bHRpLXNwZWFrZXIgc2NyaXB0IC0+IGF1ZGlvCmV4cG9ydCAgICB3YXYvbXAzL3ppcCBwYWNrYWdpbmcKZW5naW5lcyAgIGVuZ2luZSByZWdpc3RyeSAodmllbmV1IC8gdm94Y3BtKSArIGhvdC1zd2FwIGxvYWRlcgoiIiIKCmZyb20gLiBpbXBvcnQgYXVkaW8sIGNvbmZpZywgZXhwb3J0LCByZW5kZXIsIHZvaWNlYmFuawpmcm9tIC5jb25maWcgaW1wb3J0IENGRywgZW5zdXJlX2RpcnMKCiMgZW5naW5lczogaW1wb3J0IG5oZSAodG9yY2gvdmllbmV1L3ZveGNwbSBjaGkgbmFwIGtoaSBsb2FkKCkpLgpmcm9tIC4gaW1wb3J0IGVuZ2luZXMgICMgbm9xYTogRTQwMgoKIyBhcHA6IGtlbyB0aGVvIGdyYWRpbyAtPiBjaGkgaW1wb3J0IGtoaSBjYW4gKHRyb25nIG5vdGVib29rL2FwcCkuCgpfX2FsbF9fID0gWwogICAgImNvbmZpZyIsCiAgICAiYXVkaW8iLAogICAgInZvaWNlYmFuayIsCiAgICAicmVuZGVyIiwKICAgICJleHBvcnQiLAogICAgImVuZ2luZXMiLAogICAgIkNGRyIsCiAgICAiZW5zdXJlX2RpcnMiLApdCg==",
"studio/config.py": "IiIiQ2F1IGhpbmggdHJ1bmcgdGFtIFZvaWNlIFN0dWRpbyBBSS4gRG9pIENGRyBvIGRheSBsYSBkb2kgdG9hbiBibyBub3RlYm9vay4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwoKIyAtLS0gRHVvbmcgZGFuIHRyZW4gS2FnZ2xlIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KV09SSyA9IG9zLmVudmlyb24uZ2V0KCJWT0lDRV9TVFVESU9fV09SSyIsICIva2FnZ2xlL3dvcmtpbmcvdm9pY2Vfc3R1ZGlvIikKCkNGRzogZGljdCA9IHsKICAgICMgRW5naW5lIGRhbmcgc3UgZHVuZzogInZpZW5ldSIgfCAidm94Y3BtIiB8ICJjaGF0dGVyYm94IiB8IC4uLgogICAgImVuZ2luZSI6ICJ2aWVuZXUiLAogICAgIyAiY3VkYSIgfCAiY3B1Ii4gVHJlbiBLYWdnbGUgY2hvbiBBY2NlbGVyYXRvciA9IEdQVSBUNCB4Mi4KICAgICJkZXZpY2UiOiAiY3VkYSIsCiAgICAjIFRodSBtdWMgZ29jIGNodWEgbW9pIHNhbiBwaGFtLgogICAgIndvcmtfZGlyIjogV09SSywKICAgICMgS2hvIGdpb25nICh2b2ljZSBiYW5rKSAtIFRSVSBDT1Q6IHZ1b3QgcXVhIHNlc3Npb24gS2FnZ2xlLgogICAgImJhbmtfZGlyIjogb3MucGF0aC5qb2luKFdPUkssICJ2b2ljZWJhbmsiKSwKICAgICMgU2FuIHBoYW0gc2luaCByYSAod2F2L21wMykuCiAgICAib3V0X2RpciI6IG9zLnBhdGguam9pbihXT1JLLCAib3V0cHV0IiksCiAgICAjIFRoYW0gc28gc2luaCBhbSBtYWMgZGluaC4KICAgICJzYW1wbGVfcmF0ZSI6IDQ4MDAwLAogICAgImNmZ192YWx1ZSI6IDIuMCwgICAgICAgICAgICAjIGNsYXNzaWZpZXItZnJlZSBndWlkYW5jZSAoVm94Q1BNMikKICAgICJpbmZlcmVuY2VfdGltZXN0ZXBzIjogMTAsICAgIyBDRk0gc3RlcHMgKFZveENQTTIpCiAgICAiZGVmYXVsdF9zdHlsZSI6ICJkb2NfdHJ1eWVuIiwgICMgVmllTmV1OiBuYXR1cmFsIHwgbmV3cyB8IGRvY190cnV5ZW4KICAgICMgR2lvaSBoYW4gZG8gZGFpIGZpbGUgZGUgZ2hpIHJhIChnaWF5KSAtIGJhbyB2ZSBzZXNzaW9uLgogICAgIm1heF9zZWNvbmRzIjogNjAwLAp9CgojIFRlbiBlbmdpbmUgaGllbiB0aGkgKyBnaGkgY2h1LgpFTkdJTkVfSU5GTzogZGljdCA9IHsKICAgICJ2aWVuZXUiOiB7CiAgICAgICAgImxhYmVsIjogIlZpZU5ldSB2My1UdXJibyAoVGllbmcgVmlldCkiLAogICAgICAgICJsaWNlbnNlIjogIkFwYWNoZS0yLjAiLAogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly9naXRodWIuY29tL3BubmJhbzk3L1ZpZU5ldS1UVFMiLAogICAgICAgICJub3RlcyI6ICIyMyBnaW9uZyAzIG1pZW4sIGNsb25lIDMtOHMsIGVtb3Rpb24gY3VlcywgcG9kY2FzdCBtb2RlLCA0OGtIeiwgfjFHQiBWUkFNIiwKICAgIH0sCiAgICAidm94Y3BtIjogewogICAgICAgICJsYWJlbCI6ICJWb3hDUE0yIChEZXNpZ24gKyBDbG9uZSArIFN0eWxlKSIsCiAgICAgICAgImxpY2Vuc2UiOiAiQXBhY2hlLTIuMCIsCiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL2dpdGh1Yi5jb20vT3BlbkJNQi9Wb3hDUE0iLAogICAgICAgICJub3RlcyI6ICJWb2ljZSBEZXNpZ24gdHUgbW8gdGEsIENvbnRyb2xsYWJsZS9VbHRpbWF0ZSBDbG9uaW5nLCAzMCBuZ29uIG5ndSAoY28gVFYpLCA0OGtIeiwgfjhHQiBWUkFNIiwKICAgIH0sCiAgICAiY2hhdHRlcmJveCI6IHsKICAgICAgICAibGFiZWwiOiAiQ2hhdHRlcmJveCBNdWx0aWxpbmd1YWwgKHJvYWRtYXApIiwKICAgICAgICAibGljZW5zZSI6ICJNSVQiLAogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly9naXRodWIuY29tL3Jlc2VtYmxlLWFpL2NoYXR0ZXJib3giLAogICAgICAgICJub3RlcyI6ICIyMyBuZ29uIG5ndSwgemVyby1zaG90IGNsb25lLCBwYXJhbGluZ3Vpc3RpYyB0YWdzIC0gZGFuZyByb2FkbWFwIiwKICAgIH0sCn0KCgpkZWYgZW5zdXJlX2RpcnMoKSAtPiBOb25lOgogICAgIiIiVGFvIHRodSBtdWMgbGFtIHZpZWMuIEdvaSAxIGxhbiBraGkga2hvaSBkb25nLiIiIgogICAgZm9yIGsgaW4gKCJ3b3JrX2RpciIsICJiYW5rX2RpciIsICJvdXRfZGlyIik6CiAgICAgICAgb3MubWFrZWRpcnMoQ0ZHW2tdLCBleGlzdF9vaz1UcnVlKQo=",
"studio/audio.py": "IiIiVGllbiBpY2ggYW0gdGhhbmg6IGRvYy9naGksIHJlc2FtcGxlLCBtb25vLCB0cmltIHNpbGVudCwgY2h1YW4gaG9hIGFtIGx1b25nLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNzCmZyb20gdHlwaW5nIGltcG9ydCBUdXBsZQoKaW1wb3J0IG51bXB5IGFzIG5wCgp0cnk6CiAgICBpbXBvcnQgc291bmRmaWxlIGFzIHNmCmV4Y2VwdCBFeGNlcHRpb246ICAjIHByYWdtYTogbm8gY292ZXIKICAgIHNmID0gTm9uZQoKdHJ5OgogICAgaW1wb3J0IGxpYnJvc2EKZXhjZXB0IEV4Y2VwdGlvbjogICMgcHJhZ21hOiBubyBjb3ZlcgogICAgbGlicm9zYSA9IE5vbmUKCgpkZWYgbG9hZF9hdWRpbyhwYXRoOiBzdHIsIHRhcmdldF9zcjogaW50IHwgTm9uZSA9IE5vbmUpIC0+IFR1cGxlW25wLm5kYXJyYXksIGludF06CiAgICAiIiJEb2MgZmlsZSBhbSB0aGFuaCAtPiAoZmxvYXQzMiBtb25vIFstMSwxXSwgc2FtcGxlX3JhdGUpLiIiIgogICAgaWYgc2YgaXMgbm90IE5vbmU6CiAgICAgICAgZGF0YSwgc3IgPSBzZi5yZWFkKHBhdGgsIGR0eXBlPSJmbG9hdDMyIiwgYWx3YXlzXzJkPVRydWUpCiAgICAgICAgd2F2ID0gZGF0YS5tZWFuKGF4aXM9MSkKICAgIGVsaWYgbGlicm9zYSBpcyBub3QgTm9uZToKICAgICAgICB3YXYsIHNyID0gbGlicm9zYS5sb2FkKHBhdGgsIHNyPXRhcmdldF9zciwgbW9ubz1UcnVlKQogICAgZWxzZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNhbiBgc291bmRmaWxlYCBob2FjIGBsaWJyb3NhYCBkZSBkb2MgYW0gdGhhbmguIikKICAgIGlmIHRhcmdldF9zciBhbmQgc3IgIT0gdGFyZ2V0X3NyOgogICAgICAgIHdhdiA9IHJlc2FtcGxlKHdhdiwgc3IsIHRhcmdldF9zcikKICAgICAgICBzciA9IHRhcmdldF9zcgogICAgcmV0dXJuIHdhdi5hc3R5cGUobnAuZmxvYXQzMiksIGludChzcikKCgpkZWYgcmVzYW1wbGUod2F2OiBucC5uZGFycmF5LCBzcjogaW50LCB0YXJnZXRfc3I6IGludCkgLT4gbnAubmRhcnJheToKICAgIGlmIHNyID09IHRhcmdldF9zcjoKICAgICAgICByZXR1cm4gd2F2CiAgICBpZiBsaWJyb3NhIGlzIG5vdCBOb25lOgogICAgICAgIHJldHVybiBsaWJyb3NhLnJlc2FtcGxlKHdhdiwgb3JpZ19zcj1zciwgdGFyZ2V0X3NyPXRhcmdldF9zcikKICAgICMgRmFsbGJhY2s6IGxpbmVhciBpbnRlcnBvbGF0aW9uIChrZW0gY2hhdCBsdW9uZyBob24sIGNoaSBkdW5nIGtoaSB0aGlldSBsaWJyb3NhKS4KICAgIG4gPSBpbnQocm91bmQobGVuKHdhdikgKiB0YXJnZXRfc3IgLyBzcikpCiAgICByZXR1cm4gbnAuaW50ZXJwKG5wLmxpbnNwYWNlKDAsIGxlbih3YXYpIC0gMSwgbiksIG5wLmFyYW5nZShsZW4od2F2KSksIHdhdikuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIHRvX21vbm8od2F2OiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgcmV0dXJuIHdhdiBpZiB3YXYubmRpbSA9PSAxIGVsc2Ugd2F2Lm1lYW4oYXhpcz0xKS5hc3R5cGUobnAuZmxvYXQzMikKCgpkZWYgdHJpbV9zaWxlbmNlKHdhdjogbnAubmRhcnJheSwgc3I6IGludCwgdG9wX2RiOiBmbG9hdCA9IDM1LjApIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJDYXQgYm90IGtob2FuZyBsYW5nIG8gMiBkYXUuIEdpdSBhbSB0aGFuaCBub2kuIiIiCiAgICBpZiBsaWJyb3NhIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHdhdgogICAgaWR4ID0gbGlicm9zYS5lZmZlY3RzLnNwbGl0KHdhdiwgdG9wX2RiPXRvcF9kYikKICAgIGlmIGxlbihpZHgpID09IDA6CiAgICAgICAgcmV0dXJuIHdhdgogICAgcmV0dXJuIHdhdltpZHhbMF1bMF0gOiBpZHhbLTFdWzFdXQoKCmRlZiBub3JtYWxpemUod2F2OiBucC5uZGFycmF5LCBwZWFrOiBmbG9hdCA9IDAuOTcpIC0+IG5wLm5kYXJyYXk6CiAgICBtID0gZmxvYXQobnAubWF4KG5wLmFicyh3YXYpKSkgaWYgd2F2LnNpemUgZWxzZSAwLjAKICAgIGlmIG0gPCAxZS05OgogICAgICAgIHJldHVybiB3YXYKICAgIHJldHVybiAod2F2ICogKHBlYWsgLyBtKSkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIHNhdmVfd2F2KHBhdGg6IHN0ciwgd2F2OiBucC5uZGFycmF5LCBzcjogaW50KSAtPiBzdHI6CiAgICBvcy5tYWtlZGlycyhvcy5wYXRoLmRpcm5hbWUocGF0aCkgb3IgIi4iLCBleGlzdF9vaz1UcnVlKQogICAgaWYgc2YgaXMgTm9uZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNhbiBgc291bmRmaWxlYCBkZSBnaGkgd2F2LiIpCiAgICBzZi53cml0ZShwYXRoLCBucC5jbGlwKHdhdiwgLTEuMCwgMS4wKSwgc3IpCiAgICByZXR1cm4gcGF0aAoKCmRlZiBzYXZlX21wMyhwYXRoOiBzdHIsIHdhdjogbnAubmRhcnJheSwgc3I6IGludCwgYml0cmF0ZTogc3RyID0gIjE5MmsiKSAtPiBzdHI6CiAgICAiIiJXQVYgLT4gTVAzIHF1YSBmZm1wZWcgKG1hYyBkaW5oIGNvIHRyZW4gS2FnZ2xlKS4iIiIKICAgIHRtcCA9IHBhdGggKyAiLnRtcC53YXYiCiAgICBzYXZlX3dhdih0bXAsIHdhdiwgc3IpCiAgICB0cnk6CiAgICAgICAgc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgIFsiZmZtcGVnIiwgIi15IiwgIi1sb2dsZXZlbCIsICJlcnJvciIsICItaSIsIHRtcCwgIi1iOmEiLCBiaXRyYXRlLCBwYXRoXSwKICAgICAgICAgICAgY2hlY2s9VHJ1ZSwKICAgICAgICApCiAgICBmaW5hbGx5OgogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKHRtcCk6CiAgICAgICAgICAgIG9zLnJlbW92ZSh0bXApCiAgICByZXR1cm4gcGF0aAoKCmRlZiBjb25jYXQod2F2czogbGlzdCwgc3I6IGludCwgZ2FwX3NlY29uZHM6IGZsb2F0ID0gMC4yNSkgLT4gbnAubmRhcnJheToKICAgICIiIk5vaSBuaGlldSBkb2FuIGFtIHRoYW5oIGxhaSwgY2hlbiBraG9hbmcgbGFuZyBnaXVhIGNhYyBkb2FuLiIiIgogICAgZ2FwID0gbnAuemVyb3MoaW50KHNyICogZ2FwX3NlY29uZHMpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgcGFydHMgPSBbXQogICAgZm9yIGksIHcgaW4gZW51bWVyYXRlKHdhdnMpOgogICAgICAgIHBhcnRzLmFwcGVuZChucC5hc2FycmF5KHcsIGR0eXBlPW5wLmZsb2F0MzIpKQogICAgICAgIGlmIGkgPCBsZW4od2F2cykgLSAxOgogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoZ2FwKQogICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKHBhcnRzKSBpZiBwYXJ0cyBlbHNlIG5wLnplcm9zKDAsIGR0eXBlPW5wLmZsb2F0MzIpCg==",
"studio/voicebank.py": "IiIiVk9JQ0UgQkFOSyAtIFRSVSBDT1QgY3VhIFZvaWNlIFN0dWRpby4KCkthZ2dsZSBzZXNzaW9uIGxhIHRhbSB0aG9pIChoZXQgc2Vzc2lvbiA9IG1hdCBmaWxlKS4gVmkgdmF5IG1vaSBnaW9uZyBkdW9jCnRoaWV0IGtlL2Nsb25lIGRldSBkdW9jIGx1dSB0aGFuaCAqdm9pY2UgcHJvZmlsZSogdmEgZXhwb3J0IHJhIC56aXAgZGUgdGFpIHZlLAppbXBvcnQgbGFpIGtoaSBtbyBzZXNzaW9uIG1vaS4gRGF5IGxhIHRpbmggbmFuZyAidHVvbmcgbGFpIGtob25nIHRoZSB0aGlldSIKbWEgaGF1IGhldCBub3RlYm9vayBUVFMgdHJlbiBLYWdnbGUgYm8gcXVhLgoKTW90IHByb2ZpbGUgbHV1OiB0ZW4sIGxvYWkgKHByZXNldHxjbG9uZXxkZXNpZ24pLCBlbmdpbmUsIGZpbGUgbWF1IChuZXUgY2xvbmUpLAptbyB0YSAobmV1IGRlc2lnbiksIHRyYW5zY3JpcHQgbWF1LCBzdHlsZSBtYWMgZGluaC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHNodXRpbAppbXBvcnQgemlwZmlsZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcywgZmllbGQKZnJvbSB0eXBpbmcgaW1wb3J0IExpc3QsIE9wdGlvbmFsCgpNRVRBX1ZFUlNJT04gPSAxCgoKQGRhdGFjbGFzcwpjbGFzcyBWb2ljZVByb2ZpbGU6CiAgICBuYW1lOiBzdHIKICAgIGtpbmQ6IHN0ciAgICAgICAgICAgICAgICAgICAgICAgIyAicHJlc2V0IiB8ICJjbG9uZSIgfCAiZGVzaWduIgogICAgZW5naW5lOiBzdHIgICAgICAgICAgICAgICAgICAgICAjICJ2aWVuZXUiIHwgInZveGNwbSIgfCAuLi4KICAgIHJlZl9hdWRpbzogc3RyID0gIiIgICAgICAgICAgICAgIyBkdW9uZyBkYW4gZmlsZSBtYXUgKGNsb25lKSAtIGx1dSBiZW4gdHJvbmcgYmFuawogICAgcmVmX3RleHQ6IHN0ciA9ICIiICAgICAgICAgICAgICAjIHRyYW5zY3JpcHQgY3VhIGZpbGUgbWF1ICh1bHRpbWF0ZSBjbG9uaW5nKQogICAgZGVzYzogc3RyID0gIiIgICAgICAgICAgICAgICAgICAjIG1vIHRhIGdpb25nIChkZXNpZ24pCiAgICBzdHlsZTogc3RyID0gIiIgICAgICAgICAgICAgICAgICMgc3R5bGUgbWFjIGRpbmggKG5hdHVyYWx8bmV3c3xkb2NfdHJ1eWVufC4uLikKICAgIG1ldGE6IGRpY3QgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKCiAgICBkZWYgdG9fZGljdChzZWxmKSAtPiBkaWN0OgogICAgICAgIHJldHVybiBhc2RpY3Qoc2VsZikKCgpkZWYgX3Byb2ZpbGVfZGlyKGJhbmtfZGlyOiBzdHIsIG5hbWU6IHN0cikgLT4gc3RyOgogICAgc2FmZSA9ICIiLmpvaW4oYyBpZiBjLmlzYWxudW0oKSBvciBjIGluICItXyAiIGVsc2UgIl8iIGZvciBjIGluIG5hbWUpLnN0cmlwKCkgb3IgInZvaWNlIgogICAgcmV0dXJuIG9zLnBhdGguam9pbihiYW5rX2Rpciwgc2FmZSkKCgpkZWYgc2F2ZV9wcm9maWxlKGJhbmtfZGlyOiBzdHIsIHByb2ZpbGU6IFZvaWNlUHJvZmlsZSwgcmVmX3NyYzogT3B0aW9uYWxbc3RyXSA9IE5vbmUpIC0+IHN0cjoKICAgICIiIkx1dSBwcm9maWxlICsgY29weSBmaWxlIG1hdSAobmV1IGNvKSB2YW8gYmFuay4gVHJhIHZlIHRodSBtdWMgcHJvZmlsZS4iIiIKICAgIHBkaXIgPSBfcHJvZmlsZV9kaXIoYmFua19kaXIsIHByb2ZpbGUubmFtZSkKICAgIG9zLm1ha2VkaXJzKHBkaXIsIGV4aXN0X29rPVRydWUpCgogICAgaWYgcmVmX3NyYyBhbmQgb3MucGF0aC5leGlzdHMocmVmX3NyYyk6CiAgICAgICAgZXh0ID0gb3MucGF0aC5zcGxpdGV4dChyZWZfc3JjKVsxXSBvciAiLndhdiIKICAgICAgICBkc3QgPSBvcy5wYXRoLmpvaW4ocGRpciwgInJlZiIgKyBleHQpCiAgICAgICAgc2h1dGlsLmNvcHkyKHJlZl9zcmMsIGRzdCkKICAgICAgICBwcm9maWxlLnJlZl9hdWRpbyA9ICJyZWYiICsgZXh0ICAjIGR1b25nIGRhbiB0dW9uZyBkb2kgYmVuIHRyb25nIHByb2ZpbGUKCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKHBkaXIsICJwcm9maWxlLmpzb24iKSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChwcm9maWxlLnRvX2RpY3QoKSwgZiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikKICAgIHJldHVybiBwZGlyCgoKZGVmIGxvYWRfcHJvZmlsZShiYW5rX2Rpcjogc3RyLCBuYW1lOiBzdHIpIC0+IFZvaWNlUHJvZmlsZToKICAgIHBkaXIgPSBfcHJvZmlsZV9kaXIoYmFua19kaXIsIG5hbWUpCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKHBkaXIsICJwcm9maWxlLmpzb24iKSwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBkID0ganNvbi5sb2FkKGYpCiAgICBwcm9mID0gVm9pY2VQcm9maWxlKCoqe2s6IGQuZ2V0KGssIGdldGF0dHIoVm9pY2VQcm9maWxlLCBrLCAiIikpIGZvciBrIGluIGR9KQogICAgaWYgcHJvZi5yZWZfYXVkaW8gYW5kIG5vdCBvcy5wYXRoLmlzYWJzKHByb2YucmVmX2F1ZGlvKToKICAgICAgICBwcm9mLnJlZl9hdWRpbyA9IG9zLnBhdGguam9pbihwZGlyLCBwcm9mLnJlZl9hdWRpbykKICAgIHJldHVybiBwcm9mCgoKZGVmIGxpc3RfcHJvZmlsZXMoYmFua19kaXI6IHN0cikgLT4gTGlzdFtzdHJdOgogICAgaWYgbm90IG9zLnBhdGguaXNkaXIoYmFua19kaXIpOgogICAgICAgIHJldHVybiBbXQogICAgb3V0ID0gW10KICAgIGZvciBlbnRyeSBpbiBzb3J0ZWQob3MubGlzdGRpcihiYW5rX2RpcikpOgogICAgICAgIGlmIG9zLnBhdGguaXNmaWxlKG9zLnBhdGguam9pbihiYW5rX2RpciwgZW50cnksICJwcm9maWxlLmpzb24iKSk6CiAgICAgICAgICAgIG91dC5hcHBlbmQoZW50cnkpCiAgICByZXR1cm4gb3V0CgoKZGVmIGRlbGV0ZV9wcm9maWxlKGJhbmtfZGlyOiBzdHIsIG5hbWU6IHN0cikgLT4gYm9vbDoKICAgIHBkaXIgPSBfcHJvZmlsZV9kaXIoYmFua19kaXIsIG5hbWUpCiAgICBpZiBvcy5wYXRoLmlzZGlyKHBkaXIpOgogICAgICAgIHNodXRpbC5ybXRyZWUocGRpcikKICAgICAgICByZXR1cm4gVHJ1ZQogICAgcmV0dXJuIEZhbHNlCgoKZGVmIGV4cG9ydF9iYW5rKGJhbmtfZGlyOiBzdHIsIHppcF9wYXRoOiBzdHIpIC0+IHN0cjoKICAgICIiIkRvbmcgdG9hbiBibyBraG8gZ2lvbmcgdGhhbmggLnppcCBkZSB0YWkgdmUgdnVvdCBxdWEgc2Vzc2lvbi4iIiIKICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZSh6aXBfcGF0aCkgb3IgIi4iLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoemlwX3BhdGgsICJ3IiwgemlwZmlsZS5aSVBfREVGTEFURUQpIGFzIHo6CiAgICAgICAgZm9yIG5hbWUgaW4gbGlzdF9wcm9maWxlcyhiYW5rX2Rpcik6CiAgICAgICAgICAgIHBkaXIgPSBfcHJvZmlsZV9kaXIoYmFua19kaXIsIG5hbWUpCiAgICAgICAgICAgIGZvciBmbiBpbiBvcy5saXN0ZGlyKHBkaXIpOgogICAgICAgICAgICAgICAgei53cml0ZShvcy5wYXRoLmpvaW4ocGRpciwgZm4pLCBhcmNuYW1lPW9zLnBhdGguam9pbihuYW1lLCBmbikpCiAgICByZXR1cm4gemlwX3BhdGgKCgpkZWYgaW1wb3J0X2JhbmsoemlwX3BhdGg6IHN0ciwgYmFua19kaXI6IHN0cikgLT4gTGlzdFtzdHJdOgogICAgIiIiTmFwIC56aXAga2hvIGdpb25nIHZhbyBiYW5rLiBUcmEgdmUgZGFuaCBzYWNoIHRlbiBkYSBpbXBvcnQuIiIiCiAgICBpbXBvcnRlZCA9IFtdCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZSh6aXBfcGF0aCwgInIiKSBhcyB6OgogICAgICAgIHouZXh0cmFjdGFsbChiYW5rX2RpcikKICAgICAgICBmb3IgaW5mbyBpbiB6LmluZm9saXN0KCk6CiAgICAgICAgICAgIHRvcCA9IGluZm8uZmlsZW5hbWUuc3BsaXQoIi8iKVswXQogICAgICAgICAgICBpZiB0b3AgYW5kIHRvcCBub3QgaW4gaW1wb3J0ZWQ6CiAgICAgICAgICAgICAgICBpbXBvcnRlZC5hcHBlbmQodG9wKQogICAgcmV0dXJuIFtwIGZvciBwIGluIGltcG9ydGVkIGlmIG9zLnBhdGguaXNmaWxlKG9zLnBhdGguam9pbihiYW5rX2RpciwgcCwgInByb2ZpbGUuanNvbiIpKV0K",
"studio/render.py": "IiIiUmVuZGVyIHNjcmlwdCBkYSBnaW9uZyAocG9kY2FzdCAvIGF1ZGlvYm9vayAvIGhvaSB0aG9haSkgLT4gYW0gdGhhbmguCgpNb2kgZG9uZyB0cm9uZyBzY3JpcHQgPSB7dm9pY2UsIHRleHQsIHN0eWxlfS4gRW5naW5lIHNpbmggdHVuZyBkb25nIHJvaSBub2kgbGFpLgpDYWNoIG5heSBob2F0IGRvbmcgdHJlbiBNT0kgZW5naW5lIChraG9uZyBwaHUgdGh1b2MgQVBJIHJpZW5nKSwgdmkgdmF5IGRlCiJsYXAgcmFwIHRoZW0gZW5naW5lIGtoYWMiIG1hIGtob25nIHBoYWkgc3VhIHJlbmRlci4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKZnJvbSB0eXBpbmcgaW1wb3J0IENhbGxhYmxlLCBMaXN0CgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLiBpbXBvcnQgYXVkaW8KZnJvbSAuY29uZmlnIGltcG9ydCBDRkcKCiMgSGFtIHNpbmggYW0gY2h1YW4gaG9hOiAodGV4dCwgdm9pY2Vfb3JfcmVmLCBzdHlsZSkgLT4gKHdhdiwgc3IpClN5bnRoRm4gPSBDYWxsYWJsZVtbc3RyLCBzdHIsIHN0cl0sIHR1cGxlXQoKCmRlZiByZW5kZXJfbGluZXMoc3ludGg6IFN5bnRoRm4sIGxpbmVzOiBMaXN0W2RpY3RdLCBnYXBfc2Vjb25kczogZmxvYXQgPSAwLjI1KSAtPiB0dXBsZToKICAgICIiIlNpbmggdHVuZyBkb25nIHZhIG5vaSBsYWkgLT4gKHdhdiwgc3IpLiBsaW5lcyA9IFt7dm9pY2UsIHRleHQsIHN0eWxlfV0uIiIiCiAgICB3YXZzOiBMaXN0W25wLm5kYXJyYXldID0gW10KICAgIHNyID0gQ0ZHWyJzYW1wbGVfcmF0ZSJdCiAgICBmb3IgaSwgbG4gaW4gZW51bWVyYXRlKGxpbmVzKToKICAgICAgICB0ZXh0ID0gKGxuLmdldCgidGV4dCIpIG9yICIiKS5zdHJpcCgpCiAgICAgICAgaWYgbm90IHRleHQ6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdywgcyA9IHN5bnRoKHRleHQsIGxuLmdldCgidm9pY2UiLCAiIiksIGxuLmdldCgic3R5bGUiLCAiIikpCiAgICAgICAgdyA9IGF1ZGlvLm5vcm1hbGl6ZShhdWRpby50b19tb25vKG5wLmFzYXJyYXkodywgZHR5cGU9bnAuZmxvYXQzMikpKQogICAgICAgIHNyID0gaW50KHMpCiAgICAgICAgd2F2cy5hcHBlbmQodykKICAgIG1lcmdlZCA9IGF1ZGlvLmNvbmNhdCh3YXZzLCBzciwgZ2FwX3NlY29uZHM9Z2FwX3NlY29uZHMpCiAgICByZXR1cm4gbWVyZ2VkLCBzcgoKCmRlZiBzYXZlX3NjcmlwdChvdXRfZGlyOiBzdHIsIG5hbWU6IHN0ciwgd2F2OiBucC5uZGFycmF5LCBzcjogaW50LAogICAgICAgICAgICAgICAgcGVyX2xpbmU6IGJvb2wgPSBGYWxzZSwgd2F2czogTGlzdFtucC5uZGFycmF5XSB8IE5vbmUgPSBOb25lKSAtPiBMaXN0W3N0cl06CiAgICAiIiJMdXUgc2NyaXB0IGRhIHJlbmRlciB0aGFuaCB3YXYuIFRyYSB2ZSBkYW5oIHNhY2ggZHVvbmcgZGFuIGZpbGUuIiIiCiAgICBvcy5tYWtlZGlycyhvdXRfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgcGF0aHMgPSBbXQogICAgbWFpbiA9IG9zLnBhdGguam9pbihvdXRfZGlyLCBmIntuYW1lfS53YXYiKQogICAgYXVkaW8uc2F2ZV93YXYobWFpbiwgd2F2LCBzcikKICAgIHBhdGhzLmFwcGVuZChtYWluKQogICAgaWYgcGVyX2xpbmUgYW5kIHdhdnM6CiAgICAgICAgc3ViID0gb3MucGF0aC5qb2luKG91dF9kaXIsIGYie25hbWV9X2xpbmVzIikKICAgICAgICBvcy5tYWtlZGlycyhzdWIsIGV4aXN0X29rPVRydWUpCiAgICAgICAgZm9yIGksIHcgaW4gZW51bWVyYXRlKHdhdnMsIDEpOgogICAgICAgICAgICBwID0gb3MucGF0aC5qb2luKHN1YiwgZiJ7aTowM2R9LndhdiIpCiAgICAgICAgICAgIGF1ZGlvLnNhdmVfd2F2KHAsIHcsIHNyKQogICAgICAgICAgICBwYXRocy5hcHBlbmQocCkKICAgIHJldHVybiBwYXRocwo=",
"studio/export.py": "IiIiRG9uZyBnb2kgc2FuIHBoYW06IGdoaSB3YXYvbXAzLCB0YW8gemlwIHRhaSB2ZS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwppbXBvcnQgemlwZmlsZQpmcm9tIHR5cGluZyBpbXBvcnQgTGlzdAoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC4gaW1wb3J0IGF1ZGlvCgoKZGVmIHdyaXRlX291dHB1dChwYXRoOiBzdHIsIHdhdjogbnAubmRhcnJheSwgc3I6IGludCwgZm10OiBzdHIgPSAid2F2IikgLT4gc3RyOgogICAgaWYgZm10ID09ICJtcDMiOgogICAgICAgIHJldHVybiBhdWRpby5zYXZlX21wMyhwYXRoIGlmIHBhdGguZW5kc3dpdGgoIi5tcDMiKSBlbHNlIHBhdGggKyAiLm1wMyIsIHdhdiwgc3IpCiAgICByZXR1cm4gYXVkaW8uc2F2ZV93YXYocGF0aCBpZiBwYXRoLmVuZHN3aXRoKCIud2F2IikgZWxzZSBwYXRoICsgIi53YXYiLCB3YXYsIHNyKQoKCmRlZiB6aXBfb3V0cHV0cyhvdXRfZGlyOiBzdHIsIHppcF9wYXRoOiBzdHIsIHBhdHRlcm5zOiB0dXBsZSA9ICgiLndhdiIsICIubXAzIikpIC0+IHN0cjoKICAgICIiIk5lbiB0b2FuIGJvIHNhbiBwaGFtIHRyb25nIG91dF9kaXIgdGhhbmggLnppcCBkZSB0YWkgMSBsYW4uIiIiCiAgICBvcy5tYWtlZGlycyhvcy5wYXRoLmRpcm5hbWUoemlwX3BhdGgpIG9yICIuIiwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKHppcF9wYXRoLCAidyIsIHppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyB6OgogICAgICAgIGZvciBmbiBpbiBzb3J0ZWQob3MubGlzdGRpcihvdXRfZGlyKSk6CiAgICAgICAgICAgIGlmIGZuLmxvd2VyKCkuZW5kc3dpdGgocGF0dGVybnMpOgogICAgICAgICAgICAgICAgei53cml0ZShvcy5wYXRoLmpvaW4ob3V0X2RpciwgZm4pLCBhcmNuYW1lPWZuKQogICAgcmV0dXJuIHppcF9wYXRoCgoKZGVmIGxpc3Rfb3V0cHV0cyhvdXRfZGlyOiBzdHIpIC0+IExpc3Rbc3RyXToKICAgIGlmIG5vdCBvcy5wYXRoLmlzZGlyKG91dF9kaXIpOgogICAgICAgIHJldHVybiBbXQogICAgcmV0dXJuIFsKICAgICAgICBvcy5wYXRoLmpvaW4ob3V0X2RpciwgZikKICAgICAgICBmb3IgZiBpbiBzb3J0ZWQob3MubGlzdGRpcihvdXRfZGlyKSkKICAgICAgICBpZiBmLmxvd2VyKCkuZW5kc3dpdGgoKCIud2F2IiwgIi5tcDMiKSkKICAgIF0K",
"studio/engines.py": "77u/IiIiRW5naW5lIFRUUyArIGhvdC1zd2FwLgoKTmd1eWVuIHRhYzogY2hpIG5hcCBEVU5HIE1PVCBlbmdpbmUgdmFvIFZSQU0gbW9pIGx1YyAoaG90LXN3YXApIGRlIGtob25nIE9PTQp0cmVuIEthZ2dsZSBUNC4gTXVvbiB0aGVtIGVuZ2luZSBtb2kgLT4ga2UgdGh1YSBCYXNlRW5naW5lIHZhIGRhbmcga3kgdmFvIFJFR0lTVFJZCigibGFwIHJhcCBkdW9jIHRoZW0gY2FpIGtoYWMgdmFvIikuCgpNb2kgZW5naW5lIGNodWFuIGhvYSB2ZSAxIGhhbToKICAgIHR0cyh0ZXh0LCAqLCBkZXNjPSIiLCByZWZfcGF0aD0iIiwgcmVmX3RleHQ9IiIsIHByZXNldD0iIiwgc3R5bGU9IiIpIC0+ICh3YXYsIHNyKQotIGRlc2MgICAgIDogbW8gdGEgZ2lvbmcgKFZPSUNFIERFU0lHTiAtIHRhbyBnaW9uZyBtb2kgdHUgY2h1KQotIHJlZl9wYXRoIDogZmlsZSBtYXUgKFZPSUNFIENMT05FKQotIHJlZl90ZXh0IDogdHJhbnNjcmlwdCBjdWEgZmlsZSBtYXUgKHVsdGltYXRlIGNsb25pbmcsIHR1eSBjaG9uKQotIHByZXNldCAgIDogdGVuIGdpb25nIHNhbiBjbyAoZ2FsbGVyeSkKLSBzdHlsZSAgICA6IGh1b25nIGRpZW4gKHRvYyBkbywgY2FtIHh1YywgZG9jIHRydXllbi4uLikKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgZ2MKZnJvbSB0eXBpbmcgaW1wb3J0IExpc3QsIE9wdGlvbmFsLCBUdXBsZQoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC4gaW1wb3J0IGF1ZGlvCmZyb20gLmNvbmZpZyBpbXBvcnQgQ0ZHCgpXYXZPdXQgPSBUdXBsZVtucC5uZGFycmF5LCBpbnRdCgoKY2xhc3MgQmFzZUVuZ2luZToKICAgIG5hbWU6IHN0ciA9ICJiYXNlIgogICAgbGFiZWw6IHN0ciA9ICIiCiAgICBsaWNlbnNlOiBzdHIgPSAiIgogICAgc291cmNlOiBzdHIgPSAiIgoKICAgIGRlZiBsb2FkKHNlbGYpIC0+IE5vbmU6IC4uLgogICAgZGVmIHVubG9hZChzZWxmKSAtPiBOb25lOgogICAgICAgIGdjLmNvbGxlY3QoKQogICAgICAgIHRyeToKICAgICAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCgogICAgQHByb3BlcnR5CiAgICBkZWYgc2FtcGxlX3JhdGUoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBDRkdbInNhbXBsZV9yYXRlIl0KCiAgICBkZWYgdHRzKHNlbGYsIHRleHQ6IHN0ciwgKiwgZGVzYzogc3RyID0gIiIsIHJlZl9wYXRoOiBzdHIgPSAiIiwKICAgICAgICAgICAgcmVmX3RleHQ6IHN0ciA9ICIiLCBwcmVzZXQ6IHN0ciA9ICIiLCBzdHlsZTogc3RyID0gIiIpIC0+IFdhdk91dDoKICAgICAgICByYWlzZSBOb3RJbXBsZW1lbnRlZEVycm9yCgogICAgZGVmIGxpc3RfcHJlc2V0cyhzZWxmKSAtPiBMaXN0W3N0cl06CiAgICAgICAgcmV0dXJuIFtdCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIFZpZU5ldSAtLS0KY2xhc3MgVmllbmV1RW5naW5lKEJhc2VFbmdpbmUpOgogICAgIiIiVmllTmV1LVRUUyB2My1UdXJibyAtIHRpZW5nIFZpZXQgc28gMS4gQXBhY2hlLTIuMC4iIiIKCiAgICBuYW1lID0gInZpZW5ldSIKICAgIGxhYmVsID0gIlZpZU5ldSB2My1UdXJibyAoVGllbmcgVmlldCkiCiAgICBsaWNlbnNlID0gIkFwYWNoZS0yLjAiCiAgICBzb3VyY2UgPSAiaHR0cHM6Ly9naXRodWIuY29tL3BubmJhbzk3L1ZpZU5ldS1UVFMiCgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5fdHRzID0gTm9uZQoKICAgIGRlZiBsb2FkKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgc2VsZi5fdHRzIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4KICAgICAgICBmcm9tIHZpZW5ldSBpbXBvcnQgVmllbmV1ICAjIG5vcWEKCiAgICAgICAgc2VsZi5fdHRzID0gVmllbmV1KCkgICMgbWFjIGRpbmggdjMtVHVyYm8sIHR1IGNodXllbiBQeVRvcmNoIHRyZW4gR1BVCgogICAgZGVmIHVubG9hZChzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuX3R0cyA9IE5vbmUKICAgICAgICBzdXBlcigpLnVubG9hZCgpCgogICAgZGVmIGxpc3RfcHJlc2V0cyhzZWxmKSAtPiBMaXN0W3N0cl06CiAgICAgICAgc2VsZi5sb2FkKCkKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJldHVybiBsaXN0KHNlbGYuX3R0cy5saXN0X3ByZXNldF92b2ljZXMoKSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICByZXR1cm4gW10KCiAgICBkZWYgdHRzKHNlbGYsIHRleHQ6IHN0ciwgKiwgZGVzYzogc3RyID0gIiIsIHJlZl9wYXRoOiBzdHIgPSAiIiwKICAgICAgICAgICAgcmVmX3RleHQ6IHN0ciA9ICIiLCBwcmVzZXQ6IHN0ciA9ICIiLCBzdHlsZTogc3RyID0gIiIpIC0+IFdhdk91dDoKICAgICAgICBzZWxmLmxvYWQoKQogICAgICAgIHN0eWxlID0gc3R5bGUgb3IgQ0ZHWyJkZWZhdWx0X3N0eWxlIl0KICAgICAgICAjIEVtb3Rpb24gY3VlczogZ2FuIFtjdW9pXSwgW3RobyBkYWldIHRydWMgdGllcCB2YW8gdGV4dCBsYSBkdW9jICh2My1UdXJibykuCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpZiByZWZfcGF0aDoKICAgICAgICAgICAgICAgIHdhdiA9IHNlbGYuX3R0cy5pbmZlcih0ZXh0LCByZWZfYXVkaW89cmVmX3BhdGgsIGRlbm9pc2U9VHJ1ZSwgc3R5bGU9c3R5bGUpCiAgICAgICAgICAgIGVsaWYgcHJlc2V0OgogICAgICAgICAgICAgICAgd2F2ID0gc2VsZi5fdHRzLmluZmVyKHRleHQsIHZvaWNlPXByZXNldCwgc3R5bGU9c3R5bGUpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICB3YXYgPSBzZWxmLl90dHMuaW5mZXIodGV4dCwgc3R5bGU9c3R5bGUpICAjIGdpb25nIG1hYyBkaW5oCiAgICAgICAgZXhjZXB0IFR5cGVFcnJvcjoKICAgICAgICAgICAgIyBQaGllbiBiYW4gU0RLIGtob25nIG5oYW4gc3R5bGU9IC0+IGdvaSBsYWkgdGh1YW4uCiAgICAgICAgICAgIGlmIHJlZl9wYXRoOgogICAgICAgICAgICAgICAgd2F2ID0gc2VsZi5fdHRzLmluZmVyKHRleHQsIHJlZl9hdWRpbz1yZWZfcGF0aCwgZGVub2lzZT1UcnVlKQogICAgICAgICAgICBlbGlmIHByZXNldDoKICAgICAgICAgICAgICAgIHdhdiA9IHNlbGYuX3R0cy5pbmZlcih0ZXh0LCB2b2ljZT1wcmVzZXQpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICB3YXYgPSBzZWxmLl90dHMuaW5mZXIodGV4dCkKICAgICAgICByZXR1cm4gX3RvX21vbm9fd2F2KHdhdiksIHNlbGYuc2FtcGxlX3JhdGUKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gVm94Q1BNIC0tLQpjbGFzcyBWb3hDUE1FbmdpbmUoQmFzZUVuZ2luZSk6CiAgICAiIiJWb3hDUE0yIC0gRGVzaWduICsgQ29udHJvbGxhYmxlL1VsdGltYXRlIENsb25pbmcgKyBTdHlsZS4gQXBhY2hlLTIuMC4iIiIKCiAgICBuYW1lID0gInZveGNwbSIKICAgIGxhYmVsID0gIlZveENQTTIgKERlc2lnbiArIENsb25lICsgU3R5bGUpIgogICAgbGljZW5zZSA9ICJBcGFjaGUtMi4wIgogICAgc291cmNlID0gImh0dHBzOi8vZ2l0aHViLmNvbS9PcGVuQk1CL1ZveENQTSIKCiAgICBkZWYgX19pbml0X18oc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLl9tb2RlbCA9IE5vbmUKCiAgICBkZWYgbG9hZChzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYuX21vZGVsIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4KICAgICAgICBmcm9tIHZveGNwbSBpbXBvcnQgVm94Q1BNICAjIG5vcWEKCiAgICAgICAgc2VsZi5fbW9kZWwgPSBWb3hDUE0uZnJvbV9wcmV0cmFpbmVkKCJvcGVuYm1iL1ZveENQTTIiLCBsb2FkX2Rlbm9pc2VyPUZhbHNlKQoKICAgIGRlZiB1bmxvYWQoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLl9tb2RlbCA9IE5vbmUKICAgICAgICBzdXBlcigpLnVubG9hZCgpCgogICAgZGVmIHR0cyhzZWxmLCB0ZXh0OiBzdHIsICosIGRlc2M6IHN0ciA9ICIiLCByZWZfcGF0aDogc3RyID0gIiIsCiAgICAgICAgICAgIHJlZl90ZXh0OiBzdHIgPSAiIiwgcHJlc2V0OiBzdHIgPSAiIiwgc3R5bGU6IHN0ciA9ICIiKSAtPiBXYXZPdXQ6CiAgICAgICAgc2VsZi5sb2FkKCkKICAgICAgICBrdyA9IGRpY3QoY2ZnX3ZhbHVlPUNGR1siY2ZnX3ZhbHVlIl0sIGluZmVyZW5jZV90aW1lc3RlcHM9Q0ZHWyJpbmZlcmVuY2VfdGltZXN0ZXBzIl0pCiAgICAgICAgaWYgZGVzYyBhbmQgcmVmX3BhdGg6CiAgICAgICAgICAgICMgQ29udHJvbGxhYmxlIGNsb25pbmc6IHJlZiBxdXlldCBkaW5oICJhaSBub2kiLCAoc3R5bGUrZGVzYykgcXV5ZXQgZGluaCAibm9pIHRoZSBuYW8iLgogICAgICAgICAgICBwcm9tcHQgPSBmIih7ZGVzY30pIHt0ZXh0fSIgaWYgZGVzYyBlbHNlIHRleHQKICAgICAgICAgICAgaWYgc3R5bGU6CiAgICAgICAgICAgICAgICBwcm9tcHQgPSBmIih7c3R5bGV9KSB7cHJvbXB0fSIKICAgICAgICAgICAgd2F2ID0gc2VsZi5fbW9kZWwuZ2VuZXJhdGUodGV4dD1wcm9tcHQsIHJlZmVyZW5jZV93YXZfcGF0aD1yZWZfcGF0aCwgKiprdykKICAgICAgICBlbGlmIGRlc2M6CiAgICAgICAgICAgICMgVm9pY2UgZGVzaWduOiB0YW8gZ2lvbmcgbW9pIHR1IG1vIHRhLCBraG9uZyBjYW4gcmVmLgogICAgICAgICAgICB3YXYgPSBzZWxmLl9tb2RlbC5nZW5lcmF0ZSh0ZXh0PWYiKHtkZXNjfSkge3RleHR9IiwgKiprdykKICAgICAgICBlbGlmIHJlZl9wYXRoIGFuZCByZWZfdGV4dDoKICAgICAgICAgICAgIyBVbHRpbWF0ZSBjbG9uaW5nOiByZWYgKyB0cmFuc2NyaXB0IC0+IHRydW5nIHRodWMgbmhhdC4KICAgICAgICAgICAgd2F2ID0gc2VsZi5fbW9kZWwuZ2VuZXJhdGUodGV4dD10ZXh0LCBwcm9tcHRfd2F2X3BhdGg9cmVmX3BhdGgsIHByb21wdF90ZXh0PXJlZl90ZXh0LCAqKmt3KQogICAgICAgIGVsaWYgcmVmX3BhdGg6CiAgICAgICAgICAgICMgUmVmZXJlbmNlLW9ubHkgY2xvbmluZy4KICAgICAgICAgICAgcHJvbXB0ID0gZiIoe3N0eWxlfSkge3RleHR9IiBpZiBzdHlsZSBlbHNlIHRleHQKICAgICAgICAgICAgd2F2ID0gc2VsZi5fbW9kZWwuZ2VuZXJhdGUodGV4dD1wcm9tcHQsIHJlZmVyZW5jZV93YXZfcGF0aD1yZWZfcGF0aCwgKiprdykKICAgICAgICBlbHNlOgogICAgICAgICAgICB3YXYgPSBzZWxmLl9tb2RlbC5nZW5lcmF0ZSh0ZXh0PXRleHQsICoqa3cpCiAgICAgICAgc3IgPSBnZXRhdHRyKHNlbGYuX21vZGVsLnR0c19tb2RlbCwgInNhbXBsZV9yYXRlIiwgc2VsZi5zYW1wbGVfcmF0ZSkKICAgICAgICByZXR1cm4gX3RvX21vbm9fd2F2KHdhdiksIGludChzcikKCgpkZWYgX3RvX21vbm9fd2F2KHdhdikgLT4gbnAubmRhcnJheToKICAgIGlmIGlzaW5zdGFuY2Uod2F2LCAobGlzdCwgdHVwbGUpKToKICAgICAgICB3YXYgPSB3YXZbMF0KICAgIGFyciA9IG5wLmFzYXJyYXkod2F2LCBkdHlwZT1ucC5mbG9hdDMyKQogICAgcmV0dXJuIGF1ZGlvLm5vcm1hbGl6ZShhdWRpby50b19tb25vKGFycikpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIFJlZ2lzdHJ5IC0tClJFR0lTVFJZID0gewogICAgInZpZW5ldSI6IFZpZW5ldUVuZ2luZSwKICAgICJ2b3hjcG0iOiBWb3hDUE1FbmdpbmUsCn0KCl9DVVJSRU5UOiBPcHRpb25hbFtCYXNlRW5naW5lXSA9IE5vbmUKCgpkZWYgYXZhaWxhYmxlX2VuZ2luZXMoKSAtPiBMaXN0W3N0cl06CiAgICByZXR1cm4gbGlzdChSRUdJU1RSWS5rZXlzKCkpCgoKZGVmIGdldF9lbmdpbmUobmFtZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUsIHN3YXA6IGJvb2wgPSBUcnVlKSAtPiBCYXNlRW5naW5lOgogICAgIiIiTGF5IGVuZ2luZSAobWFjIGRpbmg6IGVuZ2luZSBkYW5nIGNob24pLiBzd2FwPVRydWUgLT4gdGhheSBlbmdpbmUgY3UsIGdpYWkgcGhvbmcgVlJBTS4iIiIKICAgIGdsb2JhbCBfQ1VSUkVOVAogICAgbmFtZSA9IG5hbWUgb3IgQ0ZHWyJlbmdpbmUiXQogICAgaWYgbmFtZSBub3QgaW4gUkVHSVNUUlk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkVuZ2luZSAne25hbWV9JyBjaHVhIGR1b2MgZGFuZyBreS4gQ286IHthdmFpbGFibGVfZW5naW5lcygpfSIpCiAgICBpZiBfQ1VSUkVOVCBpcyBub3QgTm9uZSBhbmQgX0NVUlJFTlQubmFtZSA9PSBuYW1lIGFuZCBub3Qgc3dhcDoKICAgICAgICByZXR1cm4gX0NVUlJFTlQKICAgIGlmIHN3YXAgYW5kIF9DVVJSRU5UIGlzIG5vdCBOb25lIGFuZCBfQ1VSUkVOVC5uYW1lICE9IG5hbWU6CiAgICAgICAgX0NVUlJFTlQudW5sb2FkKCkKICAgICAgICBfQ1VSUkVOVCA9IE5vbmUKICAgIGlmIF9DVVJSRU5UIGlzIE5vbmU6CiAgICAgICAgX0NVUlJFTlQgPSBSRUdJU1RSWVtuYW1lXSgpCiAgICAgICAgX0NVUlJFTlQubG9hZCgpCiAgICByZXR1cm4gX0NVUlJFTlQK",
"studio/app.py": "IiIiVm9pY2UgU3R1ZGlvIC0gZGlldSBwaG9pIGNhcCBjYW8gKyBHcmFkaW8gU3R1ZGlvIFVJICg1IHRhYikuCgpUYWI6CiAgMS4gRGVzaWduICAgLSB0YW8gZ2lvbmcgTU9JIHR1IG1vIHRhIGJhbmcgY2h1IChraG9uZyBjYW4gY2xpcCkKICAyLiBDbG9uZSAgICAtIG5oYW4gYmFuIGdpb25nIHR1IGNsaXAgMy0xNXMgKyBkaWV1IGtoaWVuIHN0eWxlCiAgMy4gQmFuayAgICAgLSBraG8gZ2lvbmc6IHhlbS9kb2kveG9hLCBleHBvcnQvaW1wb3J0IC56aXAgKHZ1b3Qgc2Vzc2lvbikKICA0LiBTY3JpcHQgICAtIHNjcmlwdCBkYSBnaW9uZyAtPiBwb2RjYXN0IC8gYXVkaW9ib29rIC8gaG9pIHRob2FpCiAgNS4gRXhwb3J0ICAgLSB0YWkgc2FuIHBoYW0gKHdhdi9tcDMpICsgbmVuIHppcAoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwppbXBvcnQgdGltZQpmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgVHVwbGUKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuIGltcG9ydCBhdWRpbywgZXhwb3J0IGFzIGV4LCByZW5kZXIKZnJvbSAuY29uZmlnIGltcG9ydCBDRkcsIEVOR0lORV9JTkZPLCBlbnN1cmVfZGlycwpmcm9tIC5lbmdpbmVzIGltcG9ydCBnZXRfZW5naW5lCmZyb20gLnZvaWNlYmFuayBpbXBvcnQgKAogICAgVm9pY2VQcm9maWxlLAogICAgZGVsZXRlX3Byb2ZpbGUsCiAgICBleHBvcnRfYmFuaywKICAgIGltcG9ydF9iYW5rLAogICAgbGlzdF9wcm9maWxlcywKICAgIGxvYWRfcHJvZmlsZSwKICAgIHNhdmVfcHJvZmlsZSwKKQoKV2F2T3V0ID0gVHVwbGVbbnAubmRhcnJheSwgaW50XQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHN5bnRoIC0tCmRlZiBzeW50aF90ZXh0KHRleHQ6IHN0ciwgbW9kZTogc3RyLCBkZXNjOiBzdHIsIHJlZl9wYXRoOiBzdHIsIHJlZl90ZXh0OiBzdHIsCiAgICAgICAgICAgICAgIHByZXNldDogc3RyLCBzdHlsZTogc3RyKSAtPiBXYXZPdXQ6CiAgICAiIiJTaW5oIGFtIHRoZW8gY2hlIGRvIHRydWMgdGllcCAoa2hvbmcgcXVhIHByb2ZpbGUpLiIiIgogICAgZW5nID0gZ2V0X2VuZ2luZSgpCiAgICByZXR1cm4gZW5nLnR0cygKICAgICAgICB0ZXh0LAogICAgICAgIGRlc2M9ZGVzYyBpZiBtb2RlID09ICJkZXNpZ24iIGVsc2UgIiIsCiAgICAgICAgcmVmX3BhdGg9cmVmX3BhdGggaWYgbW9kZSA9PSAiY2xvbmUiIGVsc2UgIiIsCiAgICAgICAgcmVmX3RleHQ9cmVmX3RleHQsCiAgICAgICAgcHJlc2V0PXByZXNldCBpZiBtb2RlID09ICJwcmVzZXQiIGVsc2UgIiIsCiAgICAgICAgc3R5bGU9c3R5bGUsCiAgICApCgoKZGVmIHN5bnRoX3Byb2ZpbGUocHJvZmlsZV9uYW1lOiBzdHIsIHRleHQ6IHN0ciwgc3R5bGU6IHN0ciA9ICIiKSAtPiBXYXZPdXQ6CiAgICAiIiJTaW5oIGFtIGJhbmcgbW90IGdpb25nIGRhIGx1dSB0cm9uZyBiYW5rLiIiIgogICAgZW5nID0gZ2V0X2VuZ2luZSgpCiAgICBwcm9mID0gbG9hZF9wcm9maWxlKENGR1siYmFua19kaXIiXSwgcHJvZmlsZV9uYW1lKQogICAgc3QgPSBzdHlsZSBvciBwcm9mLnN0eWxlCiAgICBpZiBwcm9mLmtpbmQgPT0gImRlc2lnbiI6CiAgICAgICAgcmV0dXJuIGVuZy50dHModGV4dCwgZGVzYz1wcm9mLmRlc2MsIHN0eWxlPXN0KQogICAgaWYgcHJvZi5raW5kID09ICJjbG9uZSI6CiAgICAgICAgcmV0dXJuIGVuZy50dHModGV4dCwgcmVmX3BhdGg9cHJvZi5yZWZfYXVkaW8sIHJlZl90ZXh0PXByb2YucmVmX3RleHQsIHN0eWxlPXN0KQogICAgcmV0dXJuIGVuZy50dHModGV4dCwgcHJlc2V0PXByb2YubmFtZSwgc3R5bGU9c3QpCgoKZGVmIF9zYXZlX291dCh3YXY6IG5wLm5kYXJyYXksIHNyOiBpbnQsIHN0ZW06IHN0ciwgZm10OiBzdHIgPSAid2F2IikgLT4gc3RyOgogICAgZW5zdXJlX2RpcnMoKQogICAgcGF0aCA9IG9zLnBhdGguam9pbihDRkdbIm91dF9kaXIiXSwgZiJ7c3RlbX1fe2ludCh0aW1lLnRpbWUoKSl9LntmbXR9IikKICAgIHJldHVybiBleC53cml0ZV9vdXRwdXQocGF0aCwgd2F2LCBzciwgZm10PWZtdCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIFVJIC0tLQpkZWYgYnVpbGRfdWkoKToKICAgIGltcG9ydCBncmFkaW8gYXMgZ3IKCiAgICBlbnN1cmVfZGlycygpCgogICAgZGVmIF9uYW1lcygpOgogICAgICAgIHJldHVybiBsaXN0X3Byb2ZpbGVzKENGR1siYmFua19kaXIiXSkKCiAgICBkZWYgX3ByZXNldHMoKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJldHVybiBnZXRfZW5naW5lKCkubGlzdF9wcmVzZXRzKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICByZXR1cm4gW10KCiAgICAjIC0tLS0gaGFuZGxlcnMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBkb19kZXNpZ24oZGVzYywgdGV4dCwgc3R5bGUsIG5hbWUsIGZtdCk6CiAgICAgICAgaWYgbm90IHRleHQuc3RyaXAoKToKICAgICAgICAgICAgcmFpc2UgZ3IuRXJyb3IoIk5oYXAgbm9pIGR1bmcgY2FuIGRvYy4iKQogICAgICAgIHdhdiwgc3IgPSBzeW50aF90ZXh0KHRleHQsICJkZXNpZ24iLCBkZXNjLCAiIiwgIiIsICIiLCBzdHlsZSkKICAgICAgICBwYXRoID0gX3NhdmVfb3V0KHdhdiwgc3IsICJkZXNpZ24iLCBmbXQpCiAgICAgICAgc2F2ZWQgPSAiIgogICAgICAgIGlmIG5hbWUuc3RyaXAoKToKICAgICAgICAgICAgc2F2ZV9wcm9maWxlKENGR1siYmFua19kaXIiXSwgVm9pY2VQcm9maWxlKAogICAgICAgICAgICAgICAgbmFtZT1uYW1lLnN0cmlwKCksIGtpbmQ9ImRlc2lnbiIsIGVuZ2luZT1DRkdbImVuZ2luZSJdLAogICAgICAgICAgICAgICAgZGVzYz1kZXNjLnN0cmlwKCksIHN0eWxlPXN0eWxlKSwgcmVmX3NyYz1Ob25lKQogICAgICAgICAgICBzYXZlZCA9IGYiRGEgbHV1IGdpb25nICoqe25hbWUuc3RyaXAoKX0qKiB2YW8gVm9pY2UgQmFuay4iCiAgICAgICAgcmV0dXJuIHBhdGgsIHNhdmVkLCBnci51cGRhdGUoY2hvaWNlcz1fbmFtZXMoKSkKCiAgICBkZWYgZG9fY2xvbmUocmVmLCByZWZfdGV4dCwgdGV4dCwgc3R5bGUsIG5hbWUsIGZtdCk6CiAgICAgICAgaWYgcmVmIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIGdyLkVycm9yKCJUYWkgbGVuIGNsaXAgbWF1ICgzLTE1cywgc2FjaCwga2hvbmcgbmhhYykuIikKICAgICAgICBpZiBub3QgdGV4dC5zdHJpcCgpOgogICAgICAgICAgICByYWlzZSBnci5FcnJvcigiTmhhcCBub2kgZHVuZyBjYW4gZG9jLiIpCiAgICAgICAgcmVmX3BhdGggPSByZWYgaWYgaXNpbnN0YW5jZShyZWYsIHN0cikgZWxzZSByZWYubmFtZQogICAgICAgIHdhdiwgc3IgPSBzeW50aF90ZXh0KHRleHQsICJjbG9uZSIsICIiLCByZWZfcGF0aCwgcmVmX3RleHQgb3IgIiIsICIiLCBzdHlsZSkKICAgICAgICBwYXRoID0gX3NhdmVfb3V0KHdhdiwgc3IsICJjbG9uZSIsIGZtdCkKICAgICAgICBzYXZlZCA9ICIiCiAgICAgICAgaWYgbmFtZS5zdHJpcCgpOgogICAgICAgICAgICBzYXZlX3Byb2ZpbGUoQ0ZHWyJiYW5rX2RpciJdLCBWb2ljZVByb2ZpbGUoCiAgICAgICAgICAgICAgICBuYW1lPW5hbWUuc3RyaXAoKSwga2luZD0iY2xvbmUiLCBlbmdpbmU9Q0ZHWyJlbmdpbmUiXSwKICAgICAgICAgICAgICAgIHJlZl90ZXh0PXJlZl90ZXh0IG9yICIiLCBzdHlsZT1zdHlsZSksIHJlZl9zcmM9cmVmX3BhdGgpCiAgICAgICAgICAgIHNhdmVkID0gZiJEYSBsdXUgZ2lvbmcgKip7bmFtZS5zdHJpcCgpfSoqIHZhbyBWb2ljZSBCYW5rLiIKICAgICAgICByZXR1cm4gcGF0aCwgc2F2ZWQsIGdyLnVwZGF0ZShjaG9pY2VzPV9uYW1lcygpKQoKICAgIGRlZiBkb19wcmV2aWV3KHByb2ZpbGVfbmFtZSwgdGV4dCwgc3R5bGUsIGZtdCk6CiAgICAgICAgaWYgbm90IHByb2ZpbGVfbmFtZToKICAgICAgICAgICAgcmFpc2UgZ3IuRXJyb3IoIkNob24gbW90IGdpb25nIHRyb25nIGJhbmsuIikKICAgICAgICBpZiBub3QgdGV4dC5zdHJpcCgpOgogICAgICAgICAgICByYWlzZSBnci5FcnJvcigiTmhhcCBub2kgZHVuZyBjYW4gZG9jLiIpCiAgICAgICAgd2F2LCBzciA9IHN5bnRoX3Byb2ZpbGUocHJvZmlsZV9uYW1lLCB0ZXh0LCBzdHlsZSkKICAgICAgICByZXR1cm4gX3NhdmVfb3V0KHdhdiwgc3IsIGYiYmFua197cHJvZmlsZV9uYW1lfSIsIGZtdCkKCiAgICBkZWYgZG9fZGVsZXRlKHByb2ZpbGVfbmFtZSk6CiAgICAgICAgaWYgcHJvZmlsZV9uYW1lOgogICAgICAgICAgICBkZWxldGVfcHJvZmlsZShDRkdbImJhbmtfZGlyIl0sIHByb2ZpbGVfbmFtZSkKICAgICAgICByZXR1cm4gZ3IudXBkYXRlKGNob2ljZXM9X25hbWVzKCksIHZhbHVlPU5vbmUpLCBmIkRhIHhvYSAqKntwcm9maWxlX25hbWV9KiouIgoKICAgIGRlZiBkb19leHBvcnRfYmFuaygpOgogICAgICAgIGVuc3VyZV9kaXJzKCkKICAgICAgICB6cCA9IG9zLnBhdGguam9pbihDRkdbIndvcmtfZGlyIl0sICJ2b2ljZV9iYW5rLnppcCIpCiAgICAgICAgZXhwb3J0X2JhbmsoQ0ZHWyJiYW5rX2RpciJdLCB6cCkKICAgICAgICByZXR1cm4genAsIGYiRGEgZG9uZyBnb2kge2xlbihfbmFtZXMoKSl9IGdpb25nLiBUYWkgLnppcCBkZSBnaXUgbGFpIHZ1b3Qgc2Vzc2lvbi4iCgogICAgZGVmIGRvX2ltcG9ydF9iYW5rKHppcGZpbGVfb2JqKToKICAgICAgICBpZiB6aXBmaWxlX29iaiBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBnci5FcnJvcigiQ2hvbiBmaWxlIHZvaWNlX2JhbmsuemlwLiIpCiAgICAgICAgenAgPSB6aXBmaWxlX29iaiBpZiBpc2luc3RhbmNlKHppcGZpbGVfb2JqLCBzdHIpIGVsc2UgemlwZmlsZV9vYmoubmFtZQogICAgICAgIGltcG9ydGVkID0gaW1wb3J0X2JhbmsoenAsIENGR1siYmFua19kaXIiXSkKICAgICAgICByZXR1cm4gZ3IudXBkYXRlKGNob2ljZXM9X25hbWVzKCkpLCBmIkRhIGltcG9ydCB7bGVuKGltcG9ydGVkKX0gZ2lvbmcuIgoKICAgIGRlZiBkb19zY3JpcHQoc2NyaXB0LCBnYXAsIGZtdCk6CiAgICAgICAgIiIiTW9pIGRvbmc6IGBURU5fR0lPTkc6IG5vaSBkdW5nYCBob2FjIGBURU5fR0lPTkcgfCBub2kgZHVuZ2AuIiIiCiAgICAgICAgbGluZXM6IExpc3RbZGljdF0gPSBbXQogICAgICAgIGZvciByYXcgaW4gKHNjcmlwdCBvciAiIikuc3BsaXRsaW5lcygpOgogICAgICAgICAgICByYXcgPSByYXcuc3RyaXAoKQogICAgICAgICAgICBpZiBub3QgcmF3IG9yIHJhdy5zdGFydHN3aXRoKCIvLyIpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgIjoiIGluIHJhdzoKICAgICAgICAgICAgICAgIHYsIHQgPSByYXcuc3BsaXQoIjoiLCAxKQogICAgICAgICAgICBlbGlmICJ8IiBpbiByYXc6CiAgICAgICAgICAgICAgICB2LCB0ID0gcmF3LnNwbGl0KCJ8IiwgMSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHYsIHQgPSAiIiwgcmF3CiAgICAgICAgICAgIGxpbmVzLmFwcGVuZCh7InZvaWNlIjogdi5zdHJpcCgpLCAidGV4dCI6IHQuc3RyaXAoKSwgInN0eWxlIjogIiJ9KQogICAgICAgIGlmIG5vdCBsaW5lczoKICAgICAgICAgICAgcmFpc2UgZ3IuRXJyb3IoIlNjcmlwdCByb25nLiBNb2kgZG9uZzogYFRFTl9HSU9ORzogbm9pIGR1bmdgLiIpCiAgICAgICAgcHJlc2V0cyA9IF9wcmVzZXRzKCkKCiAgICAgICAgZGVmIHN5bnRoKHRleHQsIHZvaWNlLCBzdHlsZSk6CiAgICAgICAgICAgIGlmIG5vdCB2b2ljZToKICAgICAgICAgICAgICAgIHJldHVybiBzeW50aF90ZXh0KHRleHQsICJwcmVzZXQiLCAiIiwgIiIsICIiLCBwcmVzZXRzWzBdIGlmIHByZXNldHMgZWxzZSAiIiwgc3R5bGUpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHJldHVybiBzeW50aF9wcm9maWxlKHZvaWNlLCB0ZXh0LCBzdHlsZSkgICAgICAgICAgIyB0ZW4gdHJvbmcgQmFuayAvIHByZXNldAogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgcmV0dXJuIHN5bnRoX3RleHQodGV4dCwgImNsb25lIiwgIiIsIHZvaWNlLCAiIiwgIiIsIHN0eWxlKSAgIyB2b2ljZSA9IGR1b25nIGRhbiBjbGlwCgogICAgICAgIHdhdiwgc3IgPSByZW5kZXIucmVuZGVyX2xpbmVzKHN5bnRoLCBsaW5lcywgZ2FwX3NlY29uZHM9Z2FwKQogICAgICAgIHJldHVybiBfc2F2ZV9vdXQod2F2LCBzciwgInNjcmlwdCIsIGZtdCkKCiAgICBkZWYgZG9fbGlzdF9vdXRwdXRzKCk6CiAgICAgICAgcmV0dXJuIGV4Lmxpc3Rfb3V0cHV0cyhDRkdbIm91dF9kaXIiXSkKCiAgICBkZWYgZG9femlwX291dHB1dHMoKToKICAgICAgICBlbnN1cmVfZGlycygpCiAgICAgICAgenAgPSBvcy5wYXRoLmpvaW4oQ0ZHWyJ3b3JrX2RpciJdLCAiYXVkaW9fb3V0cHV0cy56aXAiKQogICAgICAgIGV4LnppcF9vdXRwdXRzKENGR1sib3V0X2RpciJdLCB6cCkKICAgICAgICByZXR1cm4genAKCiAgICAjIC0tLS0gZ2lhbyBkaWVuIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIHdpdGggZ3IuQmxvY2tzKHRpdGxlPSJWb2ljZSBTdHVkaW8gQUkiKSBhcyBkZW1vOgogICAgICAgIGdyLk1hcmtkb3duKAogICAgICAgICAgICAiIyBWb2ljZSBTdHVkaW8gQUlcbiIKICAgICAgICAgICAgZiIqKkVuZ2luZToqKiBge0NGR1snZW5naW5lJ119YCDCtyAiCiAgICAgICAgICAgIGYie0VOR0lORV9JTkZPLmdldChDRkdbJ2VuZ2luZSddLCB7fSkuZ2V0KCdub3RlcycsICcnKX1cblxuIgogICAgICAgICAgICAiRGVzaWduICh0YW8gZ2lvbmcpIMK3IENsb25lIChuaGFuIGJhbikgwrcgQmFuayAoa2hvIGdpb25nKSDCtyAiCiAgICAgICAgICAgICJTY3JpcHQgKHBvZGNhc3QpIMK3IEV4cG9ydCAodGFpIHNhbiBwaGFtKS4iCiAgICAgICAgKQoKICAgICAgICAjIDEgwrcgRGVzaWduCiAgICAgICAgd2l0aCBnci5UYWIoIjEgwrcgRGVzaWduICh0YW8gZ2lvbmcgbW9pKSIpOgogICAgICAgICAgICBkX2Rlc2MgPSBnci5UZXh0Ym94KGxhYmVsPSJNbyB0YSBnaW9uZyAodHUgZG8pIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2YWx1ZT0iR2lvbmcgbmFtIHRydW5nIG5pZW4gbWllbiBOYW0sIGFtLCBob2kga2hhbiwga2UgY2h1eWVuIGNoYW0iKQogICAgICAgICAgICBkX3RleHQgPSBnci5UZXh0Ym94KGxhYmVsPSJOb2kgZHVuZyBjYW4gZG9jIiwgbGluZXM9NCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2YWx1ZT0iWGluIGNoYW8sIGRheSBsYSBnaW9uZyBkdW9jIHRoaWV0IGtlIGhvYW4gdG9hbiB0dSBtbyB0YS4iKQogICAgICAgICAgICBkX3N0eWxlID0gZ3IuVGV4dGJveChsYWJlbD0iU3R5bGUgKHR1eSBjaG9uKSIsIHZhbHVlPSJkb2NfdHJ1eWVuIikKICAgICAgICAgICAgZF9uYW1lID0gZ3IuVGV4dGJveChsYWJlbD0iVGVuIGx1dSB2YW8gQmFuayAoZGUgdHJvbmcgbmV1IGtob25nIGx1dSkiLCB2YWx1ZT0iIikKICAgICAgICAgICAgZF9mbXQgPSBnci5SYWRpbyhbIndhdiIsICJtcDMiXSwgdmFsdWU9IndhdiIsIGxhYmVsPSJEaW5oIGRhbmciKQogICAgICAgICAgICBkX2J0biA9IGdyLkJ1dHRvbigiVGhpZXQga2UgZ2lvbmciLCB2YXJpYW50PSJwcmltYXJ5IikKICAgICAgICAgICAgZF9vdXQgPSBnci5BdWRpbyhsYWJlbD0iS2V0IHF1YSIsIHR5cGU9ImZpbGVwYXRoIikKICAgICAgICAgICAgZF9tc2cgPSBnci5NYXJrZG93bigpCgogICAgICAgICMgMiDCtyBDbG9uZQogICAgICAgIHdpdGggZ3IuVGFiKCIyIMK3IENsb25lIChuaGFuIGJhbiB0dSBjbGlwKSIpOgogICAgICAgICAgICBjX3JlZiA9IGdyLkF1ZGlvKGxhYmVsPSJDbGlwIG1hdSAoMy0xNXMsIHNhY2gsIGtob25nIG5oYWMpIiwgdHlwZT0iZmlsZXBhdGgiKQogICAgICAgICAgICBjX3JlZnRleHQgPSBnci5UZXh0Ym94KGxhYmVsPSJUcmFuc2NyaXB0IGNsaXAgKHR1eSBjaG9uLCBnaW9uZyB0cnVuZyBob24pIiwgdmFsdWU9IiIpCiAgICAgICAgICAgIGNfdGV4dCA9IGdyLlRleHRib3gobGFiZWw9Ik5vaSBkdW5nIGNhbiBkb2MiLCBsaW5lcz00LCB2YWx1ZT0iRGF5IGxhIGdpb25nIGR1b2MgbmhhbiBiYW4uIikKICAgICAgICAgICAgY19zdHlsZSA9IGdyLlRleHRib3gobGFiZWw9IlN0eWxlICh0dXkgY2hvbikiLCB2YWx1ZT0iIikKICAgICAgICAgICAgY19uYW1lID0gZ3IuVGV4dGJveChsYWJlbD0iVGVuIGx1dSB2YW8gQmFuayIsIHZhbHVlPSIiKQogICAgICAgICAgICBjX2ZtdCA9IGdyLlJhZGlvKFsid2F2IiwgIm1wMyJdLCB2YWx1ZT0id2F2IiwgbGFiZWw9IkRpbmggZGFuZyIpCiAgICAgICAgICAgIGNfYnRuID0gZ3IuQnV0dG9uKCJOaGFuIGJhbiBnaW9uZyIsIHZhcmlhbnQ9InByaW1hcnkiKQogICAgICAgICAgICBjX291dCA9IGdyLkF1ZGlvKGxhYmVsPSJLZXQgcXVhIiwgdHlwZT0iZmlsZXBhdGgiKQogICAgICAgICAgICBjX21zZyA9IGdyLk1hcmtkb3duKCkKCiAgICAgICAgIyAzIMK3IEJhbmsKICAgICAgICB3aXRoIGdyLlRhYigiMyDCtyBCYW5rIChraG8gZ2lvbmcpIik6CiAgICAgICAgICAgIHdpdGggZ3IuUm93KCk6CiAgICAgICAgICAgICAgICBiX3BpY2sgPSBnci5Ecm9wZG93bihjaG9pY2VzPV9uYW1lcygpLCBsYWJlbD0iR2lvbmcgZGEgbHV1IikKICAgICAgICAgICAgICAgIGJfdGV4dCA9IGdyLlRleHRib3gobGFiZWw9Ik5vaSBkdW5nIGNhbiBkb2MiLCB2YWx1ZT0iVGh1IGdpb25nIHRyb25nIGtoby4iKQogICAgICAgICAgICBiX3N0eWxlID0gZ3IuVGV4dGJveChsYWJlbD0iU3R5bGUgKHR1eSBjaG9uKSIsIHZhbHVlPSIiKQogICAgICAgICAgICBiX2ZtdCA9IGdyLlJhZGlvKFsid2F2IiwgIm1wMyJdLCB2YWx1ZT0id2F2IiwgbGFiZWw9IkRpbmggZGFuZyIpCiAgICAgICAgICAgIHdpdGggZ3IuUm93KCk6CiAgICAgICAgICAgICAgICBiX3ByZXYgPSBnci5CdXR0b24oIkRvYyB0aHUiLCB2YXJpYW50PSJwcmltYXJ5IikKICAgICAgICAgICAgICAgIGJfZGVsID0gZ3IuQnV0dG9uKCJYb2EgZ2lvbmciKQogICAgICAgICAgICBiX291dCA9IGdyLkF1ZGlvKGxhYmVsPSJLZXQgcXVhIiwgdHlwZT0iZmlsZXBhdGgiKQogICAgICAgICAgICBiX21zZyA9IGdyLk1hcmtkb3duKCkKICAgICAgICAgICAgZ3IuTWFya2Rvd24oIiMjIyBWdW90IHNlc3Npb24gS2FnZ2xlIikKICAgICAgICAgICAgd2l0aCBnci5Sb3coKToKICAgICAgICAgICAgICAgIGJfZXhwID0gZ3IuQnV0dG9uKCJFeHBvcnQga2hvIGdpb25nIC0+IC56aXAiKQogICAgICAgICAgICAgICAgYl9pbXAgPSBnci5CdXR0b24oIkltcG9ydCAuemlwIHZhbyBraG8iKQogICAgICAgICAgICBiX2ZpbGUgPSBnci5GaWxlKGxhYmVsPSJGaWxlIHZvaWNlX2JhbmsuemlwIiwgZmlsZV90eXBlcz1bIi56aXAiXSkKCiAgICAgICAgIyA0IMK3IFNjcmlwdAogICAgICAgIHdpdGggZ3IuVGFiKCI0IMK3IFNjcmlwdCAocG9kY2FzdCAvIGF1ZGlvYm9vaykiKToKICAgICAgICAgICAgZ3IuTWFya2Rvd24oIk1vaSBkb25nIG1vdCBjYXU6IGBURU5fR0lPTkc6IG5vaSBkdW5nYC4gRG9uZyBgLy9gIGJpIGJvIHF1YS4iKQogICAgICAgICAgICBzX3NjcmlwdCA9IGdyLlRleHRib3gobGFiZWw9IlNjcmlwdCBkYSBnaW9uZyIsIGxpbmVzPTEwLCB2YWx1ZT0oCiAgICAgICAgICAgICAgICAiTWluaCBRdcOibjogQ2jDoG8gbeG7q25nIMSR4bq/biB24bubaSBWb2ljZSBTdHVkaW8gQUkuXG4iCiAgICAgICAgICAgICAgICAiTmfhu41jIFRyw6JuOiDEkMOieSBsw6Aga+G7i2NoIGLhuqNuIG5oaeG7gXUgZ2nhu41uZywgbeG7l2kgZMOybmcgbeG7mXQgY8OidS5cbiIKICAgICAgICAgICAgICAgICJNaW5oIFF1w6JuOiBC4bqlbSBuw7p0IMSR4buDIHh14bqldCB0aMOgbmggbeG7mXQgZmlsZSDDom0gdGhhbmggZHV5IG5o4bqldC4iCiAgICAgICAgICAgICkpCiAgICAgICAgICAgIHNfZ2FwID0gZ3IuU2xpZGVyKDAuMCwgMi4wLCB2YWx1ZT0wLjMsIHN0ZXA9MC4wNSwgbGFiZWw9Iktob2FuZyBsYW5nIGdpdWEgY2F1IChnaWF5KSIpCiAgICAgICAgICAgIHNfZm10ID0gZ3IuUmFkaW8oWyJ3YXYiLCAibXAzIl0sIHZhbHVlPSJ3YXYiLCBsYWJlbD0iRGluaCBkYW5nIikKICAgICAgICAgICAgc19idG4gPSBnci5CdXR0b24oIlJlbmRlciBzY3JpcHQiLCB2YXJpYW50PSJwcmltYXJ5IikKICAgICAgICAgICAgc19vdXQgPSBnci5BdWRpbyhsYWJlbD0iS2V0IHF1YSIsIHR5cGU9ImZpbGVwYXRoIikKCiAgICAgICAgIyA1IMK3IEV4cG9ydAogICAgICAgIHdpdGggZ3IuVGFiKCI1IMK3IEV4cG9ydCAodGFpIHNhbiBwaGFtKSIpOgogICAgICAgICAgICBlX3JlZnJlc2ggPSBnci5CdXR0b24oIkxhbSBtb2kgZGFuaCBzYWNoIikKICAgICAgICAgICAgZV9saXN0ID0gZ3IuRmlsZShsYWJlbD0iU2FuIHBoYW0gZGEgdGFvIiwgZmlsZV9jb3VudD0ibXVsdGlwbGUiKQogICAgICAgICAgICBlX3ppcCA9IGdyLkJ1dHRvbigiTmVuIHRvYW4gYm8gLT4gLnppcCIsIHZhcmlhbnQ9InByaW1hcnkiKQogICAgICAgICAgICBlX291dCA9IGdyLkZpbGUobGFiZWw9IkZpbGUgLnppcCIpCgogICAgICAgIGdyLk1hcmtkb3duKAogICAgICAgICAgICAiLS0tXG4iCiAgICAgICAgICAgICIqKkRhbyBkdWMgJiBjb25zZW50OioqIGNoaSBjbG9uZSBnaW9uZyBiYW4gZHVvYyBwaGVwLiBWb2ljZSBEZXNpZ24gdGFvIGdpb25nICIKICAgICAgICAgICAgInRvbmcgaG9wIChraG9uZyBjb3B5IGFpKSBsYSBjYWNoIGFuIHRvYW4gdmUgcGhhcCBseS5cbiIKICAgICAgICAgICAgIioqTGljZW5zZSBlbmdpbmU6KiogVmllTmV1IChBcGFjaGUtMi4wKSDCtyBWb3hDUE0yIChBcGFjaGUtMi4wKS4gWGVtIFJFQURNRS4iCiAgICAgICAgKQoKICAgICAgICAjIC0tLS0gd2lyaW5nIChzYXUga2hpIHRhbyBoZXQgY29tcG9uZW50IGRlIGRvaSBkdW9jIGJfcGljaykgLS0tLQogICAgICAgIGRfYnRuLmNsaWNrKGRvX2Rlc2lnbiwgW2RfZGVzYywgZF90ZXh0LCBkX3N0eWxlLCBkX25hbWUsIGRfZm10XSwKICAgICAgICAgICAgICAgICAgICBbZF9vdXQsIGRfbXNnLCBiX3BpY2tdKQogICAgICAgIGNfYnRuLmNsaWNrKGRvX2Nsb25lLCBbY19yZWYsIGNfcmVmdGV4dCwgY190ZXh0LCBjX3N0eWxlLCBjX25hbWUsIGNfZm10XSwKICAgICAgICAgICAgICAgICAgICBbY19vdXQsIGNfbXNnLCBiX3BpY2tdKQogICAgICAgIGJfcHJldi5jbGljayhkb19wcmV2aWV3LCBbYl9waWNrLCBiX3RleHQsIGJfc3R5bGUsIGJfZm10XSwgW2Jfb3V0XSkKICAgICAgICBiX2RlbC5jbGljayhkb19kZWxldGUsIFtiX3BpY2tdLCBbYl9waWNrLCBiX21zZ10pCiAgICAgICAgYl9leHAuY2xpY2soZG9fZXhwb3J0X2JhbmssIFtdLCBbYl9maWxlLCBiX21zZ10pCiAgICAgICAgYl9pbXAuY2xpY2soZG9faW1wb3J0X2JhbmssIFtiX2ZpbGVdLCBbYl9waWNrLCBiX21zZ10pCiAgICAgICAgc19idG4uY2xpY2soZG9fc2NyaXB0LCBbc19zY3JpcHQsIHNfZ2FwLCBzX2ZtdF0sIFtzX291dF0pCiAgICAgICAgZV9yZWZyZXNoLmNsaWNrKGRvX2xpc3Rfb3V0cHV0cywgW10sIFtlX2xpc3RdKQogICAgICAgIGVfemlwLmNsaWNrKGRvX3ppcF9vdXRwdXRzLCBbXSwgW2Vfb3V0XSkKCiAgICByZXR1cm4gZGVtbwoKCmRlZiBsYXVuY2goc2hhcmU6IGJvb2wgPSBUcnVlLCAqKmt3KToKICAgIGRlbW8gPSBidWlsZF91aSgpCiAgICByZXR1cm4gZGVtby5sYXVuY2goc2hhcmU9c2hhcmUsICoqa3cpCg=="
}''')
for _rel, _b64 in _FILES.items():
    _path = os.path.join(PKG, _rel)
    os.makedirs(os.path.dirname(_path), exist_ok=True)
    with open(_path, "wb") as f:
        f.write(base64.b64decode(_b64))
sys.path.insert(0, PKG)

from studio import CFG, ensure_dirs
from studio.engines import available_engines, get_engine
from studio.config import ENGINE_INFO
ensure_dirs()
CFG["engine"] = ENGINE
print("Voice Studio đã nạp. Engine có sẵn:", available_engines())


In [ ]:
# ===== 4 · NẠP MODEL + TEST NHANH =====
eng = get_engine(ENGINE)          # hot-swap: chỉ 1 model vào VRAM
print(f"Engine: {eng.label} | {eng.sample_rate} Hz | license: {eng.license}")
print("Gọng preset:", (eng.list_presets() or ["(engine tự chọn giọng mặc định)"])[:8])

# Test nhanh 1 câu (đổi text nếu muốn)
try:
    wav, sr = eng.tts("Xin chào, đây là Voice Studio AI trên Kaggle.",
                      desc="giọng nữ miền Bắc, ấm, tự nhiên" if ENGINE == "voxcpm" else "",
                      preset=(eng.list_presets() or [""])[0] if ENGINE == "vieneu" else "")
    from studio import audio
    audio.save_wav("/kaggle/working/voice_studio/output/smoke.wav", wav, sr)
    print("✅ Test OK — đã ghi output/smoke.wav")
except Exception as e:
    print("⚠️ Test lỗi (có thể do model chưa tải xong):", e)


In [ ]:
# ===== 5 · GRADIO STUDIO (5 tab) =====
from studio.app import build_ui
demo = build_ui()
demo.launch(share=True, debug=False, height=760)
# Bấm vào đường link `*.gradio.live` hiện ra để mở giao diện.


## 📥 Giữ kho giọng qua session (Voice Bank)

Session Kaggle **tạm thời** — hết phiên là mất file. Muốn giữ giọng đã thiết kế/clone:

1. Tab **3 · Bank** → bấm **Export kho giong → .zip** → tải `voice_bank.zip` về máy.
2. Lần chạy sau (session mới, Run All) → tab **3 · Bank** → **Import .zip vào kho**.

Đây là điểm khác biệt lớn nhất so với các notebook TTS thông thường: giọng nói của bạn
trở thành **tài sản tái sử dụng**, không mất theo phiên.

**Tip clone chất lượng cao:** clip 5–15 giây, thu sát mic, phòng yên, **không nhạc**,
một người nói, đọc đúng tông bạn muốn đầu ra (vì model copy cả *cách nói*, không chỉ chất giọng).


## ⚠️ Đạo đức, consent & giấy phép

- **Chỉ clone giọng khi bạn có quyền** (giọng của chính bạn hoặc được phép rõ ràng).
- **Voice Design** tạo giọng *tổng hợp* từ mô tả — không sao chép ai — là lựa chọn an toàn
  về pháp lý và đạo đức nếu bạn cần giọng nói mới.
- Không dùng để mạo danh, lừa đảo, hoặc tạo nội dung sai sự thật.

---

## 📄 Credit & License

**Voice Studio AI** được thiết kế và dựng bởi **Elainabaka** © 2026 —
notebook tạo giọng nói cho cộng đồng trên GPU Kaggle miễn phí.

- **Code notebook này:** MIT (dùng cá nhân & thương mại, giữ nguyên credit).
- **Dựng trên model mã nguồn mở** của tác giả gốc — cảm ơn và **giữ nguyên credit của họ**:

| Thành phần | Tác giả gốc | License |
|---|---|---|
| VieNeu-TTS (v3-Turbo) | Phạm Nguyễn Ngọc Bảo & cộng sự · github.com/pnnbao97/VieNeu-TTS | Apache-2.0 |
| VoxCPM2 (OpenBMB) | github.com/OpenBMB/VoxCPM | Apache-2.0 |
| Giao diện | Gradio | Apache-2.0 |

*Cảm ơn các tác giả gốc đã mở model. Vui lòng giữ nguyên mục credit khi chia sẻ lại.*


## 🗺️ Roadmap
- [ ] Thêm engine **Chatterbox Multilingual** (MIT, paralinguistic tags) — registry đã sẵn sàng.
- [ ] Thêm engine **Qwen3-TTS-VoiceDesign** (voice design SOTA).
- [ ] Tự động ASR transcript cho clip clone (Whisper) để tăng độ giống.
- [ ] Chấm điểm độ giống giọng (ECAPA cosine) sau mỗi lần clone.
- [ ] Lưu bank thẳng lên Kaggle Dataset (tùy chọn, cần token).
